# QML kernel benchmark from metriq-gym

In [1]:
# (C) Copyright IBM 2026.
#
# This code is licensed under the Apache License, Version 2.0. You may
# obtain a copy of this license in the LICENSE.txt file in the root directory
# of this source tree or at http://www.apache.org/licenses/LICENSE-2.0.
#
# Any modifications or derivative works of this code must retain this
# copyright notice, and modified files need to carry a notice indicating
# that they have been altered from the originals.

In [1]:
import time
import numpy as np
from qiskit import *
from qiskit.quantum_info import hellinger_fidelity
from qiskit_ibm_runtime import QiskitRuntimeService, Sampler
import mthree
from mthree.utils import final_measurement_mapping

from metriq_gym.benchmarks.qml_kernel import create_inner_product_circuit

import matplotlib.pyplot as plt
%config InlineBackend.figure_format='retina'

In [2]:
import json
from datetime import datetime

def default(obj):
    if isinstance(obj, datetime):
        return {
            "__type__": "datetime",
            "value": obj.isoformat(),
        }
    raise TypeError(f"Cannot serialize {type(obj)}")

def object_hook(obj):
    if obj.get("__type__") == "datetime":
        return datetime.fromisoformat(obj["value"])
    return obj

data = {
    "name": "test",
    "created": datetime.now(),
}

In [3]:
def polarization_fidelity(success_prob, num_qubits):
    """From metriq-gym

    https://github.com/unitaryfoundation/metriq-gym/metriq_gym/benchmarks/mirror_circuits.py
    """
    baseline = 1.0 / (2**num_qubits)
    polarization = (
            (success_prob - baseline) / (1.0 - baseline)
            if ((1.0 - baseline) > 0) and ((success_prob - baseline) > 0)
            else 0.0
        )
    return polarization

In [4]:
theme_colors = ['#6929c4', '#1192e8', '#005d5d', '#9f1853', 
                '#fa4d56', '#570408', '#198038', '#002d9c',
                '#ee538b', '#b28600', '#009d9a', '#012749',
                '#8a3800', '#a56eff']

In [5]:
service = QiskitRuntimeService()

In [6]:
backend = service.backend("ibm_chicago", use_fractional_gates=True)

In [35]:
st = time.perf_counter()
qubit_list = list(range(5,105,5))
#qubit_list = [64]*10
circs = [create_inner_product_circuit(num_qubits) for num_qubits in qubit_list]
trans_circs = transpile(circs, backend, optimization_level=3)
meas_maps = [final_measurement_mapping(cc) for cc in trans_circs]
ft = time.perf_counter()
classical_time = ft - st
classical_time / 10

0.48172286590015573

In [36]:
mit = mthree.M3Mitigation(backend)

In [37]:
mit_jobs = mit.cals_from_system()

In [38]:
sampler = Sampler(backend)

/tmp/ipykernel_75843/1091040122.py:1: DeprecationWarning: The SamplerV2 class is deprecated as of qiskit-ibm-runtime 0.50.0 and will be removed no sooner than 3 months after the release date. `qiskit_ibm_runtime.executor_sampler.Sampler` provides a new Sampler implementation that runs all the pre- and post-processing in the client side. The top-level `qiskit_ibm_runtime.Sampler` import will be updated to point to the client-side sampler in an upcoming release, with the legacy Sampler being available until the deprecation is completed.
  sampler = Sampler(backend)


In [39]:
shots = 10_000

In [40]:
job = sampler.run(trans_circs, shots=shots)

In [41]:
job.job_id()

'dbqvpp6jjaos73ai1u8g'

In [42]:
counts = [res.join_data().get_counts() for res in job.result()]

In [43]:
st = time.perf_counter()
mit_counts = mit.apply_correction(counts, meas_maps)
ft = time.perf_counter()
classical_time += ft - st
classical_time / 10

1.1523256099000718

In [44]:
fids = []
polarizations = []
mit_fids = []
mit_polarizations = []
for idx, num_qubits in enumerate(qubit_list):
    ans = {'0'*num_qubits:1}
    hf = hellinger_fidelity(counts[idx], ans)
    polar = polarization_fidelity(hf, num_qubits)
    fids.append(hf)
    polarizations.append(polar)
    mit_dist = mit_counts[idx].nearest_probability_distribution()
    mit_hf = hellinger_fidelity(mit_dist, ans)
    mit_polar = polarization_fidelity(mit_hf, num_qubits)
    mit_fids.append(mit_hf)
    mit_polarizations.append(mit_polar)
    print(num_qubits, polar, mit_polar, mit_polar > 1/np.e)

5 0.9776 0.984900421793766 True
10 0.9437450635386118 0.9586386523578878 True
15 0.8932967436750389 0.9133136023398581 True
20 0.8007998100278958 0.8329831680437059 True
25 0.8036999941498051 0.8592908871126179 True
30 0.7071999997273117 0.7808226361298417 True
35 0.6881999999909292 0.7677650838437039 True
40 0.551899999999598 0.6950408092750653 True
45 0.37859999999999294 0.4787771000579107 True
50 0.30760000000001475 0.40071739379256105 True
55 0.3584000000000157 0.48657600077038765 True
60 0.3053000000000188 0.4000105122347941 True
65 0.27400000000001845 0.39197912243537675 True
70 0.17580000000001944 0.24407983310482373 False
75 0.20730000000002355 0.2962654962360922 False
80 0.24220000000002176 0.36700992852007985 False
85 0.13980000000002352 0.21140606881546395 False
90 0.13680000000002457 0.20850289293230162 False
95 0.11520000000002435 0.17905668165396085 False
100 0.11200000000002455 0.17822653866916482 False


In [45]:
results = {'name': 'qml', 'backend': backend.name,
           'backend_properties': json.dumps(backend.properties().to_dict(), default=default),
           'datetime': str(job.creation_date),
           'qubits': qubit_list, 'shots': shots, 'quantum_runtime': job.usage(), 'mit_runtime': sum(mm.usage() for mm in mit_jobs),
           'classical_runtime': classical_time,
           'fids': fids, 'mit_fids': mit_fids, 'polarizations': polarizations, 'mit_polarizations': mit_polarizations}

In [55]:
with open("qml_results.json", "w") as f:
        json.dump(results, f)

In [56]:
st = time.perf_counter()
#qubit_list = list(range(5,105,5))
qubit_list = [50]*10
circs = [create_inner_product_circuit(num_qubits) for num_qubits in qubit_list]
trans_circs = transpile(circs, backend, optimization_level=3)
meas_maps = [final_measurement_mapping(cc) for cc in trans_circs]
ft = time.perf_counter()
classical_time = ft - st
classical_time / 10

0.2867588529999921

In [57]:
job2 = sampler.run(trans_circs, shots=shots)

In [58]:
counts = [res.join_data().get_counts() for res in job2.result()]

In [59]:
st = time.perf_counter()
mit_counts = mit.apply_correction(counts, meas_maps)
ft = time.perf_counter()
classical_time += ft - st
classical_time / 10

0.3450825662999705

In [60]:
fids = []
polarizations = []
mit_fids = []
mit_polarizations = []
for idx, num_qubits in enumerate(qubit_list):
    ans = {'0'*num_qubits:1}
    hf = hellinger_fidelity(counts[idx], ans)
    polar = polarization_fidelity(hf, num_qubits)
    fids.append(hf)
    polarizations.append(polar)
    mit_dist = mit_counts[idx].nearest_probability_distribution()
    mit_hf = hellinger_fidelity(mit_dist, ans)
    mit_polar = polarization_fidelity(mit_hf, num_qubits)
    mit_fids.append(mit_hf)
    mit_polarizations.append(mit_polar)
    print(num_qubits, polar, mit_polar, mit_polar > 1/np.e)

50 0.39830000000001253 0.5200470220292535 True
50 0.3950000000000142 0.5164969933055805 True
50 0.37190000000001444 0.48553716691748194 True
50 0.37190000000001483 0.4858778787978655 True
50 0.37870000000001447 0.4944852278376364 True
50 0.3714000000000146 0.48499128854429285 True
50 0.3772000000000139 0.49272528642854113 True
50 0.3642000000000145 0.475388227636508 True
50 0.3787000000000141 0.49490738960529557 True
50 0.3737000000000141 0.4878432253191254 True


In [63]:
np.mean(mit_fids)

np.float64(0.4938299706421585)

In [64]:
np.std(mit_fids)

np.float64(0.013379110153057127)